In [ ]:
%pip install torchmetrics


In [ ]:
# -*- coding: utf-8 -*-
"""
HD GAN-based Image Inpainting with Low-Rank Prior (PyTorch)
256x256 resolution, auto-download HD dataset, prints losses
Supports checkpoint save/resume.
Includes PSNR/SSIM metrics and an overfitting graph that plots periodically.
"""

import os, random, math, glob
from dataclasses import dataclass
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import requests, zipfile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

# You may need to install torchmetrics: pip install torchmetrics
import torchmetrics

In [ ]:
# ------------------------------
# Config
# ------------------------------
@dataclass
class CFG:
    DATASET = "FOLDER"
    DATA_ROOT = ".hd_images"
    IMAGE_SIZE = 256
    BATCH_SIZE = 4
    NUM_WORKERS = 0 # Set to 2 or 4 if you have a good CPU and RAM
    LR = 2e-4
    BETAS = (0.5, 0.999)
    EPOCHS = 150
    LAM_HOLE = 6.0
    LAM_VALID = 1.0
    LAM_ADV = 0.1
    LAM_RANK = 0.0
    RANK_PATCH = 16
    RANK_NUM_PATCHES = 8
    VIS_EVERY = 500
    PLOT_EVERY = 10  # <<< NEW: Show the loss graph every 10 epochs
    SEED = 42
    CKPT_DIR = "checkpoints_hd"
    CKPT_INTERVAL = 1

CFG = CFG()

In [ ]:
# ------------------------------
# Setup
# ------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG.CKPT_DIR, exist_ok=True)
torch.manual_seed(CFG.SEED)
np.random.seed(CFG.SEED)
random.seed(CFG.SEED)

# ------------------------------
# Auto-download HD dataset
# ------------------------------
def download_hd_dataset():
    if not os.path.exists(CFG.DATA_ROOT):
        os.makedirs(CFG.DATA_ROOT, exist_ok=True)
        url = "https://www.dropbox.com/s/5e1pj9v2k1w0l6q/celebahq-resized-256x256.zip?dl=1"
        zip_path = os.path.join(CFG.DATA_ROOT, "celebahq.zip")
        print("Downloading HD dataset...")
        r = requests.get(url, stream=True)
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print("Extracting...")
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(CFG.DATA_ROOT)
        os.remove(zip_path)
        print("Dataset ready at:", CFG.DATA_ROOT)
    else:
        print("Dataset already exists at:", CFG.DATA_ROOT)

download_hd_dataset()

In [ ]:

# ------------------------------
# Mask utilities
# ------------------------------
def random_box_mask(h, w):
    mask = np.ones((h, w), np.float32)
    rect_h = random.randint(h // 4, h // 2)
    rect_w = random.randint(w // 4, w // 2)
    top = random.randint(0, h - rect_h)
    left = random.randint(0, w - rect_w)
    mask[top:top + rect_h, left:left + rect_w] = 0.0
    return mask

def random_freeform_mask(h, w, strokes=4):
    mask = np.ones((h, w), np.float32)
    for _ in range(random.randint(3, strokes + 3)):
        x, y = random.randint(0, w-1), random.randint(0, h-1)
        length = random.randint(min(h,w)//6, min(h,w)//2)
        angle = random.uniform(0, 2*math.pi)
        thickness = random.randint(12, 30)
        for i in range(length):
            xi = int(x + i*math.cos(angle))
            yi = int(y + i*math.sin(angle))
            if 0<=xi<w and 0<=yi<h:
                x1,x2 = max(0, xi-thickness//2), min(w, xi+thickness//2)
                y1,y2 = max(0, yi-thickness//2), min(h, yi+thickness//2)
                mask[y1:y2, x1:x2] = 0.0
    return mask

def make_mask(h, w):
    return random_box_mask(h, w) if random.random()<0.5 else random_freeform_mask(h, w)


In [ ]:
# ------------------------------
# Dataset
# ------------------------------
class InpaintDataset(Dataset):
    def __init__(self, root, image_size=256):
        self.paths = sorted(glob.glob(os.path.join(root, "*.jpg")))
        if len(self.paths)==0:
            sub_dir_paths = sorted(glob.glob(os.path.join(root, "*", "*.jpg")))
            self.paths = sub_dir_paths
            if len(self.paths)==0:
                 raise ValueError(f"No images found in {root} or its subdirectories")
        self.sz = image_size
        self.transform = transforms.Compose([
            transforms.Resize((image_size,image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        h,w = img.shape[1], img.shape[2]
        mask_np = make_mask(h,w)
        mask = torch.from_numpy(mask_np).unsqueeze(0).float()
        masked_img = img * mask
        inp = torch.cat([masked_img, mask], dim=0)
        return inp, img, mask


In [ ]:
# ------------------------------
# Models
# ------------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3,1,1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch,3,1,1,bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.block(x)

class Down(nn.Module):
    def __init__(self,in_ch,out_ch,use_bn=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch,out_ch,4,2,1,bias=not use_bn)]
        if use_bn: layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2,inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self,x): return self.block(x)

class Up(nn.Module):
    def __init__(self,in_ch,skip_ch,out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch,out_ch,4,2,1,bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.fuse = DoubleConv(out_ch + skip_ch, out_ch)
    def forward(self,x,skip):
        x = self.up(x)
        x = self.bn(x)
        x = self.relu(x)
        x = torch.cat([x,skip],dim=1)
        return self.fuse(x)

class UNetGen(nn.Module):
    def __init__(self,in_ch=4,out_ch=3,base=64):
        super().__init__()
        self.down1 = Down(in_ch, base, use_bn=False)
        self.down2 = Down(base, base*2)
        self.down3 = Down(base*2, base*4)
        self.down4 = Down(base*4, base*8)
        self.down5 = Down(base*8, base*8)
        self.up4 = Up(base*8, base*8, base*4)
        self.up3 = Up(base*4, base*4, base*2)
        self.up2 = Up(base*2, base*2, base)
        self.up1 = Up(base, base, base)
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(base,base,4,2,1,bias=False),
            nn.BatchNorm2d(base),
            nn.ReLU(inplace=True)
        )
        self.out = nn.Conv2d(base,out_ch,1)
    def forward(self,x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        bott = self.down5(d4)
        u4 = self.up4(bott,d4)
        u3 = self.up3(u4,d3)
        u2 = self.up2(u3,d2)
        u1 = self.up1(u2,d1)
        x = self.up0(u1)
        return torch.tanh(self.out(x))

from torch.nn.utils import spectral_norm
class PatchDiscriminator(nn.Module):
    def __init__(self,in_ch=3,base=64):
        super().__init__()
        layers = [
            spectral_norm(nn.Conv2d(in_ch,base,4,2,1)), nn.LeakyReLU(0.2,inplace=True),
            spectral_norm(nn.Conv2d(base,base*2,4,2,1)), nn.BatchNorm2d(base*2), nn.LeakyReLU(0.2,inplace=True),
            spectral_norm(nn.Conv2d(base*2,base*4,4,2,1)), nn.BatchNorm2d(base*4), nn.LeakyReLU(0.2,inplace=True),
            spectral_norm(nn.Conv2d(base*4,base*8,4,1,1)), nn.BatchNorm2d(base*8), nn.LeakyReLU(0.2,inplace=True),
            spectral_norm(nn.Conv2d(base*8,1,4,1,1))
        ]
        self.main = nn.Sequential(*layers)
    def forward(self,x): return self.main(x)


In [ ]:
# Losses
# ------------------------------
def recon_loss(pred,target,mask,lam_hole=6.0,lam_valid=1.0):
    L_hole = torch.mean(torch.abs((1-mask)*(pred-target)))
    L_valid= torch.mean(torch.abs(mask*(pred-target)))
    return lam_hole*L_hole + lam_valid*L_valid, L_hole.detach(), L_valid.detach()

def d_hinge_loss(d_real,d_fake):
    return torch.mean(F.relu(1.0-d_real)) + torch.mean(F.relu(1.0+d_fake))

def g_hinge_loss(d_fake):
    return -torch.mean(d_fake)

@torch.no_grad()
def _hole_ratio(patch_mask):
    return (1.0 - patch_mask.mean()).item()

def nuclear_norm_patches(img,mask,patch=16,num_patches=8):
    if CFG.LAM_RANK == 0:
        return torch.tensor(0.0, device=img.device)
    B,C,H,W = img.shape
    total = img.new_tensor(0.)
    count = 0
    for b in range(B):
        for _ in range(num_patches):
            y = random.randint(0,H-patch)
            x = random.randint(0,W-patch)
            pm = mask[b,:,y:y+patch,x:x+patch]
            if _hole_ratio(pm)<0.5: continue
            P = img[b,:,y:y+patch,x:x+patch]
            for c in range(C):
                svals = torch.linalg.svdvals(P[c])
                total += svals.sum()
                count +=1
    return total/count if count>0 else img.new_tensor(0.)

In [ ]:
# ------------------------------
# Visualization and Plotting
# ------------------------------
def denorm(x): return (x.clamp(-1,1)+1)*0.5

def visualize_triplet(inp4,target,mask,g_out,title="",max_display=4):
    B = min(max_display, inp4.size(0))
    I = target[:B].cpu()
    M = mask[:B].cpu()
    masked = inp4[:B,:3].cpu()
    Ihat = g_out[:B].cpu()
    comp = M*I + (1-M)*Ihat

    num_rows = B
    fig, axes = plt.subplots(num_rows, 4, figsize=(20, 5 * num_rows))
    if B == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(B):
        axes[i,0].imshow(denorm(I[i]).permute(1,2,0)); axes[i,0].set_title("Original"); axes[i,0].axis("off")
        axes[i,1].imshow(denorm(masked[i]).permute(1,2,0)); axes[i,1].set_title("Masked"); axes[i,1].axis("off")
        axes[i,2].imshow(M[i,0],cmap="gray"); axes[i,2].set_title("Mask"); axes[i,2].axis("off")
        axes[i,3].imshow(denorm(comp[i]).permute(1,2,0)); axes[i,3].set_title("Inpainted"); axes[i,3].axis("off")
    plt.suptitle(title,fontsize=24)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

def plot_losses(g_loss, d_loss, val_loss):
    """
    Plots the training and validation losses.
    """
    plt.figure(figsize=(10, 5))
    plt.title("Generator and Discriminator Loss During Training")
    plt.plot(g_loss, label="G_train")
    plt.plot(d_loss, label="D_train")
    plt.plot(val_loss, label="G_val (L1)")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
import json
import os
import glob
import numpy as np

# --- Configuration ---
# This should match the configuration in your main training script.
# Using a raw string (r"...") is important for Windows paths.
CHECKPOINT_DIR = r"D:\hello\checkpoints_hd"
LOG_FILE = "training_log.json"

def manage_training_log():
    """
    This function demonstrates the complete logic for handling the training log file.
    It finds the latest checkpoint, loads the JSON log, syncs them,
    and simulates adding new data and saving the log for every epoch.
    """
    print(f"--- JSON Log Management Script ---")
    print(f"Looking for checkpoints in: {CHECKPOINT_DIR}")

    # --- Part 1: Checkpoint Loading Logic ---
    # Find the latest checkpoint to determine which epoch we should be on.
    start_epoch = 1
    ckpt_files = sorted(
        glob.glob(os.path.join(CHECKPOINT_DIR, "ckpt_epoch_*.pth")),
        key=lambda x: int(os.path.basename(x).split('_')[-1].split('.')[0])
    )

    if ckpt_files:
        latest_ckpt = ckpt_files[-1]
        print(f"Found latest checkpoint: {os.path.basename(latest_ckpt)}")
        try:
            # The next epoch to run is one after the last saved checkpoint.
            start_epoch = int(os.path.basename(latest_ckpt).split('_')[-1].split('.')[0]) + 1
        except (ValueError, IndexError):
            print("Warning: Could not parse epoch from checkpoint filename. Assuming epoch 1.")
            start_epoch = 1
    else:
        print(f"No checkpoints found in '{CHECKPOINT_DIR}'. Starting fresh.")

    # --- Part 2: JSON Log Handling (Loading and Syncing) ---
    log_data = {}
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r') as f:
            try:
                log_data = json.load(f)
                print(f"Successfully loaded previous log data from '{LOG_FILE}'.")
                
                # <<< FIX: Detect and handle old, incompatible log file format. >>>
                if 'g_train' in log_data:
                    print("\nWarning: Detected an old and incompatible log file format.")
                    backup_filename = LOG_FILE + '.bak'
                    print(f"The old log file will be renamed to '{backup_filename}'.")
                    print("A new log file with the correct format will be created.")
                    os.rename(LOG_FILE, backup_filename)
                    log_data = {} # Reset to an empty dictionary to start fresh.

            except json.JSONDecodeError:
                print(f"Warning: '{LOG_FILE}' is corrupted or empty. Starting a new log.")
                log_data = {}
    else:
        print(f"No log file found. A new '{LOG_FILE}' will be created.")

    # Syncing: Remove any log entries that are newer than our checkpoint.
    # This line will no longer cause an error because log_data is guaranteed to have the correct format.
    current_log_epochs = sorted([int(k) for k in log_data.keys()])
    synced = False
    for epoch_key in current_log_epochs:
        if epoch_key >= start_epoch:
            del log_data[str(epoch_key)]
            synced = True
            
    if synced:
        print(f"Synced log file to match the latest checkpoint. Ready to log from epoch {start_epoch}.")

    # --- Part 3: Simulating a Training Step and Saving Every Epoch ---
    # This section simulates running for multiple epochs and saves a log entry for every single one.
    print(f"\n--- Simulating training from epoch {start_epoch} up to epoch 100 ---")
    
    # We loop through several simulated epochs
    for epoch in range(start_epoch, 101):
        
        # <<< MODIFICATION: Saving data for every epoch >>>
        print(f"--- Saving data for epoch {epoch} ---")

        # In your real script, these values would be calculated by the model.
        avg_g_loss = np.random.rand()
        avg_d_loss = np.random.rand()
        val_mean_loss = np.random.rand() / 2 # Validation loss is usually lower
        epoch_psnr = 30 + np.random.rand() * 5
        epoch_ssim = 0.85 + np.random.rand() * 0.1

        # Add the new epoch's data to our dictionary.
        log_data[str(epoch)] = {
            'g_train_loss': avg_g_loss,
            'd_train_loss': avg_d_loss,
            'val_l1_loss': val_mean_loss,
            'psnr': epoch_psnr,
            'ssim': epoch_ssim
        }

    # --- Part 4: Saving the Final JSON Log File ---
    # Write the entire updated dictionary back to the file once the simulation is done.
    try:
        with open(LOG_FILE, 'w') as f:
            json.dump(log_data, f, indent=4)
        print(f"\nSuccessfully saved a complete log for every epoch to '{LOG_FILE}'.")
    except Exception as e:
        print(f"Error saving log file: {e}")

if __name__ == "__main__":
    manage_training_log()



In [ ]:

# ------------------------------
# Training
# ------------------------------
def train():
    ds = InpaintDataset(CFG.DATA_ROOT, CFG.IMAGE_SIZE)
    n_val = max(1, int(len(ds) * 0.03))
    n_train = len(ds) - n_val
    train_ds, val_ds = random_split(ds, [n_train, n_val])

    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                              num_workers=CFG.NUM_WORKERS, drop_last=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                            num_workers=CFG.NUM_WORKERS, drop_last=False, pin_memory=True)

    G = UNetGen().to(device)
    D = PatchDiscriminator().to(device)

    # --- Metrics ---
    psnr = torchmetrics.PeakSignalNoiseRatio().to(device)
    ssim = torchmetrics.StructuralSimilarityIndexMeasure().to(device)

    g_opt = torch.optim.Adam(G.parameters(), lr=CFG.LR, betas=CFG.BETAS)
    d_opt = torch.optim.Adam(D.parameters(), lr=CFG.LR, betas=CFG.BETAS)

    # --- Loss History ---
    train_g_loss_history, train_d_loss_history, val_loss_history = [], [], []

    start_epoch = 1
    best_val = 1e9
    ckpt_files = sorted(
        glob.glob(os.path.join(CFG.CKPT_DIR, "ckpt_epoch_*.pth")),
        key=lambda x: int(os.path.basename(x).split('_')[-1].split('.')[0])
    )

    if ckpt_files:
        latest_ckpt = ckpt_files[-1]
        print(f"Resuming from checkpoint {latest_ckpt}")
        checkpoint = torch.load(latest_ckpt, map_location=device)
        G.load_state_dict(checkpoint['G_state'])
        D.load_state_dict(checkpoint['D_state'])
        g_opt.load_state_dict(checkpoint['g_opt_state'])
        d_opt.load_state_dict(checkpoint['d_opt_state'])
        start_epoch = checkpoint['epoch'] + 1
        best_val = checkpoint.get('best_val', 1e9)

    step = (start_epoch - 1) * len(train_loader)
    for epoch in range(start_epoch, CFG.EPOCHS + 1):
        G.train(); D.train()
        epoch_g_loss, epoch_d_loss = 0.0, 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{CFG.EPOCHS}")
        for inp4, target, mask in pbar:
            inp4, target, mask = inp4.to(device), target.to(device), mask.to(device)

            # Discriminator
            d_opt.zero_grad()
            with torch.no_grad():
                pred = G(inp4)
                comp = mask * target + (1 - mask) * pred
            d_real, d_fake = D(target), D(comp.detach())
            d_loss = d_hinge_loss(d_real, d_fake)
            d_loss.backward()
            d_opt.step()

            # Generator
            g_opt.zero_grad()
            pred = G(inp4)
            comp = mask * target + (1 - mask) * pred
            recon, _, _ = recon_loss(pred, target, mask, CFG.LAM_HOLE, CFG.LAM_VALID)
            adv = CFG.LAM_ADV * g_hinge_loss(D(comp))
            rank = CFG.LAM_RANK * nuclear_norm_patches(pred, mask)
            g_loss = recon + adv + rank
            g_loss.backward()
            g_opt.step()

            epoch_g_loss += g_loss.item()
            epoch_d_loss += d_loss.item()
            pbar.set_postfix(g_loss=f"{g_loss.item():.4f}", d_loss=f"{d_loss.item():.4f}")

            if step % CFG.VIS_EVERY == 0:
                G.eval()
                with torch.no_grad():
                    val_inp, val_target, val_mask = next(iter(val_loader))
                    val_inp, val_target, val_mask = val_inp.to(device), val_target.to(device), val_mask.to(device)
                    out = G(val_inp)
                    visualize_triplet(val_inp, val_target, val_mask, out,
                                      title=f"Epoch {epoch}-Step {step}", max_display=2)
                G.train()
            step += 1

        # Validation
        G.eval()
        val_losses = []
        psnr.reset(); ssim.reset()
        with torch.no_grad():
            for inp4, target, mask in val_loader:
                inp4, target, mask = inp4.to(device), target.to(device), mask.to(device)
                pred = G(inp4)
                
                L_rec = recon_loss(pred, target, mask)[0]
                val_losses.append(L_rec.item())

                pred_denorm = denorm(pred)
                target_denorm = denorm(target)
                psnr.update(pred_denorm, target_denorm)
                ssim.update(pred_denorm, target_denorm)

        avg_g_loss = epoch_g_loss/len(train_loader)
        avg_d_loss = epoch_d_loss/len(train_loader)
        val_mean_loss = np.mean(val_losses)
        epoch_psnr = psnr.compute()
        epoch_ssim = ssim.compute()
        
        train_g_loss_history.append(avg_g_loss)
        train_d_loss_history.append(avg_d_loss)
        val_loss_history.append(val_mean_loss)

        print(f"Epoch {epoch} -> Train G_Loss: {avg_g_loss:.4f}, D_Loss: {avg_d_loss:.4f} | Val L1: {val_mean_loss:.4f}, PSNR: {epoch_psnr:.4f}, SSIM: {epoch_ssim:.4f}")

        # <<< MODIFIED BLOCK: Plot losses periodically >>>
        if epoch % CFG.PLOT_EVERY == 0 and epoch > 0:
            print(f"Displaying loss graph at epoch {epoch}...")
            plot_losses(train_g_loss_history, train_d_loss_history, val_loss_history)
        
        if val_mean_loss < best_val:
            best_val = val_mean_loss
            torch.save(G.state_dict(), os.path.join(CFG.CKPT_DIR, "best_G.pth"))
            torch.save(D.state_dict(), os.path.join(CFG.CKPT_DIR, "best_D.pth"))
            print("Saved best HD models.")

        if epoch % CFG.CKPT_INTERVAL == 0:
            ckpt_path = os.path.join(CFG.CKPT_DIR, f"ckpt_epoch_{epoch}.pth")
            torch.save({
                'epoch': epoch, 'G_state': G.state_dict(), 'D_state': D.state_dict(),
                'g_opt_state': g_opt.state_dict(), 'd_opt_state': d_opt.state_dict(),
                'best_val': best_val
            }, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")

    # Plot the final losses after training is complete
    print("Displaying final loss graph...")
    plot_losses(train_g_loss_history, train_d_loss_history, val_loss_history)
    return G, D

In [ ]:
# ------------------------------
# Run training
# ------------------------------
if __name__ == "__main__":
    import torch.multiprocessing as mp
    try:
        mp.set_start_method("spawn", force=True)
        print("Set multiprocessing start method to 'spawn'.")
        manage_training_log()
    except RuntimeError:
        pass
    
    print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
    G, D = train()
    print("Training finished. Best HD model saved to:", CFG.CKPT_DIR)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import math

# --- Configuration ---
LOG_FILE = "training_log.json"
INTERVAL = 10

def plot_history_intervals():
    """
    Loads training history from the new epoch-keyed format and creates detailed plots.
    """
    try:
        with open(LOG_FILE, 'r') as f:
            log_data = json.load(f)
    except FileNotFoundError:
        print(f"Error: Log file not found at '{LOG_FILE}'")
        return

    # Parse the new format: {epoch_str: {g_train_loss, d_train_loss, val_l1_loss, psnr, ssim}}
    if not log_data or not isinstance(log_data, dict):
        print("Log file is empty or invalid. No data to plot.")
        return

    # Try to extract epochs and losses from the new format
    epochs = sorted([int(k) for k in log_data.keys() if k.isdigit()])
    
    if not epochs:
        print("No valid epoch data found in log file.")
        return

    g_train_loss = [log_data[str(e)]['g_train_loss'] for e in epochs]
    d_train_loss = [log_data[str(e)]['d_train_loss'] for e in epochs]
    val_l1_loss = [log_data[str(e)]['val_l1_loss'] for e in epochs]
    psnr_vals = [log_data[str(e)].get('psnr', 0) for e in epochs]
    ssim_vals = [log_data[str(e)].get('ssim', 0) for e in epochs]

    total_epochs = len(epochs)
    print(f"Loaded {total_epochs} epochs of training data (epochs {epochs[0]}-{epochs[-1]})")

    # Use a professional plot style
    plt.style.use('seaborn-v0_8-whitegrid')

    # --- Main summary plot ---
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('HD GAN Training History (150 Epochs)', fontsize=18, fontweight='bold')

    # Plot 1: Generator and Discriminator Loss
    ax = axes[0, 0]
    ax.plot(epochs, g_train_loss, 'o-', label="G_train", color='orange', linewidth=2)
    ax.plot(epochs, d_train_loss, 's-', label="D_train", color='blue', linewidth=2)
    ax.set_title('Generator & Discriminator Loss', fontsize=14, fontweight='bold')
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)

    # Plot 2: Validation L1 Loss
    ax = axes[0, 1]
    ax.plot(epochs, val_l1_loss, 'D-', label="Val L1", color='green', linewidth=2)
    ax.fill_between(epochs, val_l1_loss, alpha=0.3, color='green')
    ax.set_title('Validation L1 Loss', fontsize=14, fontweight='bold')
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)

    # Plot 3: PSNR
    ax = axes[1, 0]
    ax.plot(epochs, psnr_vals, '^-', label="PSNR", color='red', linewidth=2)
    ax.fill_between(epochs, psnr_vals, alpha=0.3, color='red')
    ax.set_title('Peak Signal-to-Noise Ratio (PSNR)', fontsize=14, fontweight='bold')
    ax.set_xlabel("Epoch")
    ax.set_ylabel("PSNR (dB)")
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)

    # Plot 4: SSIM
    ax = axes[1, 1]
    ax.plot(epochs, ssim_vals, 'v-', label="SSIM", color='purple', linewidth=2)
    ax.fill_between(epochs, ssim_vals, alpha=0.3, color='purple')
    ax.set_title('Structural Similarity Index (SSIM)', fontsize=14, fontweight='bold')
    ax.set_xlabel("Epoch")
    ax.set_ylabel("SSIM")
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # --- Interval breakdown (optional) ---
    num_intervals = math.ceil(total_epochs / INTERVAL)
    print(f"\nGenerating {num_intervals} interval plots...")
    
    for i in range(num_intervals):
        start_idx = i * INTERVAL
        end_idx = min((i + 1) * INTERVAL, total_epochs)
        
        interval_epochs = epochs[start_idx:end_idx]
        g_interval = g_train_loss[start_idx:end_idx]
        d_interval = d_train_loss[start_idx:end_idx]
        val_interval = val_l1_loss[start_idx:end_idx]
        
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(interval_epochs, g_interval, 'o-', label="G_train", color='orange', linewidth=2)
        ax.plot(interval_epochs, d_interval, 's-', label="D_train", color='blue', linewidth=2)
        ax.plot(interval_epochs, val_interval, 'D-', label="Val L1", color='green', linewidth=2)
        
        ax.set_title(f'Training Epochs {interval_epochs[0]} to {interval_epochs[-1]}', fontsize=14)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend(fontsize=12)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    plot_history_intervals()

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import math

# --- Configuration ---
LOG_FILE = "training_log.json"
INTERVAL = 10

def plot_history_intervals():
    """
    Loads training history and creates a detailed plot for each 10-epoch
    interval, plus a final summary plot for a beautiful and clear analysis.
    """
    try:
        with open(LOG_FILE, 'r') as f:
            log_data = json.load(f)
    except FileNotFoundError:
        print(f"Error: Log file not found at '{LOG_FILE}'")
        print("Please ensure you have run the training script (`waha_modified.py`) to generate the log file.")
        return

    g_train = log_data.get('g_train', [])
    d_train = log_data.get('d_train', [])
    val_l1 = log_data.get('val_l1', [])

    if not g_train:
        print("Log file is empty. No data to plot.")
        return

    total_epochs = len(g_train)
    epochs_axis = np.arange(1, total_epochs + 1)
    
    # Use a more professional and clean plot style
    plt.style.use('seaborn-v0_8-whitegrid')

    # --- Plotting Individual Intervals ---
    num_intervals = math.ceil(total_epochs / INTERVAL)
    
    # Add 1 to num_intervals for the final summary plot
    # This creates a grid of subplots to display each interval separately
    fig, axes = plt.subplots(num_intervals + 1, 1, figsize=(12, 6 * (num_intervals + 1)), squeeze=False)
    fig.suptitle('Detailed Training Progress by Interval', fontsize=20, y=0.99)
    
    axes = axes.flatten() # Flatten to a 1D array for easy iteration

    for i in range(num_intervals):
        ax = axes[i]
        start_idx = i * INTERVAL
        end_idx = min((i + 1) * INTERVAL, total_epochs)
        
        # Get the slice of data for the current interval
        interval_epochs = epochs_axis[start_idx:end_idx]
        g_train_interval = g_train[start_idx:end_idx]
        d_train_interval = d_train[start_idx:end_idx]
        val_l1_interval = val_l1[start_idx:end_idx]
        
        ax.plot(interval_epochs, g_train_interval, 'o-', label="G_train", color='orange')
        ax.plot(interval_epochs, d_train_interval, 'o-', label="D_train", color='blue')
        ax.plot(interval_epochs, val_l1_interval, 'o-', label="G_val (L1)", color='green')
        
        ax.set_title(f'Epochs {start_idx + 1} to {end_idx}', fontsize=14)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend()
        ax.grid(True, which='both', linestyle='--', linewidth=0.5)

    # --- Plotting the Full History Summary ---
    summary_ax = axes[num_intervals]
    summary_ax.plot(epochs_axis, g_train, label="G_train", color='orange')
    summary_ax.plot(epochs_axis, d_train, label="D_train", color='blue')
    summary_ax.plot(epochs_axis, val_l1, label="G_val (L1)", color='green')
    
    # Add markers every 10 epochs to the summary plot
    marker_epochs = epochs_axis[INTERVAL-1::INTERVAL]
    g_train_markers = np.array(g_train)[INTERVAL-1::INTERVAL]
    d_train_markers = np.array(d_train)[INTERVAL-1::INTERVAL]
    val_l1_markers = np.array(val_l1)[INTERVAL-1::INTERVAL]
    
    summary_ax.scatter(marker_epochs, g_train_markers, color='darkred', zorder=5)
    summary_ax.scatter(marker_epochs, d_train_markers, color='darkred', zorder=5)
    summary_ax.scatter(marker_epochs, val_l1_markers, label=f'Interval Marker', color='darkred', zorder=5)

    summary_ax.set_title('Full Training History Summary', fontsize=16)
    summary_ax.set_xlabel("Epochs")
    summary_ax.set_ylabel("Loss")
    summary_ax.legend()
    summary_ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    summary_ax.set_xlim(left=0)
    summary_ax.set_ylim(bottom=0)

    # Hide any unused subplots from the figure
    for i in range(num_intervals + 1, len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()

# --- Run the plotting function ---
# You can run this script directly or paste its content into a Jupyter cell.
plot_history_intervals()
